# Social Influence Arena — Persona-Tuned LLM Attacker Panel

**What this notebook does**

Trains three LoRA adapters on a shared `Qwen2.5-0.5B-Instruct` base, one per adversarial persona (AUTHORITY, CONSENSUS, GASLIGHTER). The resulting adapters plug into the env's `LLMAttackerPanel` and replace the scripted-template personas with policy-parameterized learned adversaries, turning the env into a genuine multi-agent system (1 learning defender + 3 fixed-policy learned attackers).

HONEST stays template — it must deliver factually correct citations pegged to ground truth, and an LLM would risk hallucinating the answer.

**Runtime target:** Kaggle T4, ~10 min per persona ≈ 30 min total.

## 1. Install dependencies

In [ ]:
import os

# ── Portable workdir + cache setup (Kaggle / Colab / local) ─────────────
ON_KAGGLE = os.path.isdir("/kaggle/input") or os.path.isdir("/kaggle/working")
ON_COLAB  = os.path.isdir("/content")
WORKDIR = (
    "/kaggle/working" if ON_KAGGLE
    else "/content"   if ON_COLAB
    else os.path.expanduser("~/sia_work")
)
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

_CACHE = f"{WORKDIR}/hf_cache"
os.makedirs(_CACHE, exist_ok=True)
for k in ("HF_HOME", "HF_HUB_CACHE", "HUGGINGFACE_HUB_CACHE",
          "TRANSFORMERS_CACHE", "HF_DATASETS_CACHE"):
    os.environ[k] = _CACHE
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"WORKDIR={WORKDIR}; cache={_CACHE}; ON_KAGGLE={ON_KAGGLE}; ON_COLAB={ON_COLAB}")


In [ ]:
%pip install -q unsloth trl transformers accelerate datasets peft openenv-core matplotlib numpy

import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer

print("CUDA:", torch.cuda.is_available(), "Torch:", torch.__version__)

## 2. Locate the arena package and JSONL datasets

The uploaded Kaggle dataset contains the whole repo, so the persona JSONLs live under `train/attacker_data/`.

In [ ]:
# Locate and register the env package. Works on Colab (fresh / zip / clone) and Kaggle (dataset).
# Tries local sources first, then falls back to pip-install from the HF Space.

import glob, importlib, os, sys, subprocess, zipfile

ON_KAGGLE = os.path.isdir("/kaggle/input") or os.path.isdir("/kaggle/working")
EXTRACT_DIR = f"{WORKDIR}/social-influence-arena" if ON_KAGGLE else "/content/social-influence-arena"

CANDIDATE_ZIPS = [
    "/social-influence-arena.zip",
    "/content/social-influence-arena.zip",
    *glob.glob("/content/**/social-influence-arena.zip", recursive=True),
    *glob.glob("/kaggle/input/**/social-influence-arena.zip", recursive=True),
]

REPO = os.environ.get("ARENA_REPO", "")
# HF Space fallback for fresh Colab runs (no zip uploaded, no Kaggle dataset, no ARENA_REPO).
HF_SPACE_PIP = "git+https://huggingface.co/spaces/NDGCodes/social-influence-env"

def find_pkg():
    """Walk the filesystem looking for social_influence_env/__init__.py."""
    roots = ["/kaggle/input", "/kaggle/working"] if ON_KAGGLE else ["/content", "/"]
    for search_root in roots:
        if not os.path.isdir(search_root):
            continue
        for root, dirs, _ in os.walk(search_root):
            if root.startswith(("/proc", "/sys", "/dev", "/usr", "/var/cache", "/tmp/pip-")):
                dirs[:] = []
                continue
            if "social_influence_env" in dirs:
                cand = os.path.join(root, "social_influence_env")
                if os.path.isfile(os.path.join(cand, "__init__.py")):
                    return cand
    return None

# 1) Local sources first (Kaggle dataset, Colab zip drag-drop, optional git clone).
pkg_dir = find_pkg()
if pkg_dir is None:
    zip_path = next((z for z in CANDIDATE_ZIPS if os.path.isfile(z)), None)
    if zip_path:
        print(f"Extracting {zip_path} -> {EXTRACT_DIR}")
        os.makedirs(EXTRACT_DIR, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(EXTRACT_DIR)
        pkg_dir = os.path.join(EXTRACT_DIR, "envs", "social_influence_env")
        if not os.path.isfile(os.path.join(pkg_dir, "__init__.py")):
            pkg_dir = find_pkg()
    elif REPO:
        print("Cloning", REPO)
        subprocess.run(["git", "clone", REPO, EXTRACT_DIR], check=True)
        pkg_dir = find_pkg()

# 2) Fallback: pip install from the HF Space (works on fresh Colab — no upload needed).
if pkg_dir is None:
    print(f"No local source found - pip-installing from HF Space: {HF_SPACE_PIP}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", HF_SPACE_PIP])
    if "social_influence_env" in sys.modules:
        importlib.reload(sys.modules["social_influence_env"])
    import social_influence_env
    pkg_dir = os.path.dirname(social_influence_env.__file__)

assert pkg_dir is not None, "social_influence_env package not found"

parent = os.path.dirname(pkg_dir)
# repo_root is one above 'envs/' for local-source runs. For pip-installed
# packages there's no repo on disk, so fall back to WORKDIR for relative
# outputs like assets/plots/.
candidate_repo_root = os.path.dirname(parent)
repo_root = candidate_repo_root if os.path.isdir(os.path.join(candidate_repo_root, "envs")) else WORKDIR
if parent not in sys.path:
    sys.path.insert(0, parent)
os.chdir(repo_root)
print("package at:", pkg_dir)
print("cwd:       ", repo_root)

if "social_influence_env" in sys.modules:
    importlib.reload(sys.modules["social_influence_env"])
import social_influence_env
print("env imports from:", social_influence_env.__file__)


## 3. Load Qwen2.5-0.5B-Instruct + LoRA head

Same tokenizer family as the 3B defender, very fast, 4-bit ≈ 400 MB.

In [ ]:
import huggingface_hub.constants as _hf_const
import huggingface_hub.file_download as _fd
_hf_const.HF_HUB_CACHE = _CACHE
_hf_const.HUGGINGFACE_HUB_CACHE = _CACHE
_fd.HUGGINGFACE_HUB_CACHE = _CACHE

os.chdir(WORKDIR)
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

ATTACKER_BASE = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(ATTACKER_BASE, cache_dir="/tmp/huggingface", trust_remote_code=True)
model, _ = FastLanguageModel.from_pretrained(
    model_name=ATTACKER_BASE,
    max_seq_length=1536,
    load_in_4bit=True,
    cache_dir="/tmp/huggingface",
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16, lora_dropout=0.0, bias="none",
    use_gradient_checkpointing="unsloth",
)
print("base on:", model.device)

# ── Quiet down chatty transformers logging (same as defender) ───────────
try:
    model.generation_config.max_length = None
except Exception:
    pass
import warnings, logging
warnings.filterwarnings("ignore", message=r".*AttentionMaskConverter.*")
warnings.filterwarnings("ignore", message=r"Both `max_new_tokens`.*and `max_length`.*")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("transformers.generation").setLevel(logging.ERROR)
print("✅ logging quieted; generation_config.max_length cleared")


## 4. Train one LoRA adapter per persona

For each persona: load the JSONL, tokenize with the model's chat template, SFT for ~80 steps, save to `attackers/{persona}_lora/`.

We train one persona at a time from the same fresh LoRA init (`FastLanguageModel.get_peft_model` above gave us one set of LoRA weights). To train a second persona with its own adapter, we save the current weights, zero out and reinitialize the LoRA, and train the next persona.

In [ ]:
import json, shutil
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import torch

os.chdir(WORKDIR)
ATTACKERS_DIR = f"{WORKDIR}/attackers"
os.makedirs(ATTACKERS_DIR, exist_ok=True)

def load_persona_dataset(persona: str) -> Dataset:
    path = os.path.join(DATASET_DIR, f"{persona.lower()}.jsonl")
    records = []
    for line in open(path):
        rec = json.loads(line)
        text = tokenizer.apply_chat_template(rec["messages"], tokenize=False, add_generation_prompt=False)
        records.append({"text": text})
    return Dataset.from_list(records)

def snapshot_lora_init():
    """Capture the untrained LoRA weights so we can reset between personas."""
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if "lora_" in k}

def restore_lora(init_state):
    sd = model.state_dict()
    for k, v in init_state.items():
        sd[k].copy_(v.to(sd[k].device))

# Snapshot the fresh LoRA init — we'll restore before each persona's training.
_INIT = snapshot_lora_init()
print(f"Snapshotted {len(_INIT)} LoRA tensors as init.")

def train_persona(persona: str, max_steps: int = 80):
    restore_lora(_INIT)
    ds = load_persona_dataset(persona)
    eff_batch = 2 * 4  # per_device_train_batch_size * gradient_accumulation_steps
    epochs = max_steps * eff_batch / max(len(ds), 1)
    print(f"\n=== Training {persona} ===")
    print(f"  examples: {len(ds)}   max_steps: {max_steps}   "
          f"effective batch: {eff_batch}   ≈ {epochs:.1f} epochs")
    cfg = SFTConfig(
        output_dir=f"{WORKDIR}/sft_{persona.lower()}",
        max_steps=max_steps,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.05,
        max_length=1536,
        logging_steps=10,
        save_steps=9999,
        optim="adamw_8bit",
        report_to="none",
        seed=42,
        packing=False,
        dataset_text_field="text",
        dataset_num_proc=2,
    )
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, tokenizer=tokenizer)
    trainer.train()
    final_loss = trainer.state.log_history[-1].get("loss", "n/a")
    # Capture loss history for the per-persona training-curve plot below.
    LOSS_HISTORY[persona] = [
        (e["step"], e["loss"]) for e in trainer.state.log_history
        if "loss" in e and "step" in e
    ]
    out = os.path.join(ATTACKERS_DIR, f"{persona.lower()}_lora")
    if os.path.isdir(out):
        shutil.rmtree(out)
    model.save_pretrained(out)
    tokenizer.save_pretrained(out)
    print(f"  → saved {out}   (final loss ≈ {final_loss})")

LOSS_HISTORY: dict[str, list] = {}
for persona in ("AUTHORITY", "CONSENSUS", "GASLIGHTER"):
    train_persona(persona)

print("\nAll three adapters trained.")

## 4b. Training-loss curves — one per persona

Pulled from each persona's `trainer.state.log_history`. Useful both as a sanity check (loss should decrease steadily) and as a figure for the writeup.


In [ ]:
# Per-persona SFT loss curves on a single axis.
import os, matplotlib.pyplot as plt
os.makedirs("assets/plots", exist_ok=True)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for persona, hist in LOSS_HISTORY.items():
    if not hist: continue
    xs, ys = zip(*hist)
    ax.plot(xs, ys, marker="o", linewidth=1.5, label=persona)
ax.set_xlabel("step"); ax.set_ylabel("SFT loss")
ax.set_title("Attacker LoRA training — per-persona loss")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout()
out = "assets/plots/attacker_training_loss.png"
fig.savefig(out, dpi=150)
plt.show()
print(f"saved → {out}")


## 5. Zero-shot vs tuned — quick qualitative check

Swap adapters in and out of the live model and sample one attacker message per persona, against the same question. The tuned output should clearly outshine zero-shot if SFT worked.

In [ ]:
import torch
from peft import PeftModel
from social_influence_env.server.questions import ALL_QUESTIONS
from social_influence_env.server.llm_attackers import PERSONA_SYSTEM, USER_PROMPT, _format_history
from social_influence_env.models import DialogTurn, BeliefState

q = next(q for q in ALL_QUESTIONS if q.id == "m01")   # 17 + 25
fake_history = [
    DialogTurn(turn=0, persona="NEUTRAL", scenario="BASELINE",
               attacker_message="(baseline)", agent_public_response="My answer is 42.",
               agent_belief=BeliefState(answer="42", confidence=0.9, source="prior", reasoning="prior")),
]

def prompt_ids(persona):
    messages = [
        {"role": "system", "content": PERSONA_SYSTEM[persona]},
        {"role": "user", "content": USER_PROMPT.format(q=q.prompt, wrong=q.wrong_answer, history_block=_format_history(fake_history))},
    ]
    return tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)

def decode(out, ids):
    return tokenizer.decode(out[0, ids.shape[-1]:], skip_special_tokens=True).strip()

# Load each saved adapter into the model under a named slot so we can switch.
# The model currently has GASLIGHTER weights (last-trained). We reload each
# saved adapter fresh from disk to get faithful per-persona samples.
ADAPTER_NAMES = ("AUTHORITY", "CONSENSUS", "GASLIGHTER")
for persona in ADAPTER_NAMES:
    path = os.path.join(ATTACKERS_DIR, f"{persona.lower()}_lora")
    try:
        model.load_adapter(path, adapter_name=persona)
    except Exception as exc:
        print(f"[warn] {persona} load_adapter failed: {exc!r}")

for persona in ADAPTER_NAMES:
    print(f"\n=== {persona} ===")
    ids = prompt_ids(persona)

    # Zero-shot: disable all adapters → raw Qwen2.5-0.5B
    try:
        model.disable_adapter_layers()
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.9,
                                 pad_token_id=tokenizer.eos_token_id)
        print("[zero-shot]", decode(out, ids))
    except Exception as exc:
        print(f"[zero-shot] FAILED: {exc!r}")

    # Tuned: activate this persona's adapter
    try:
        model.enable_adapter_layers()
        try:
            model.set_adapter(persona)
        except Exception:
            pass
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=80, do_sample=True, temperature=0.7, top_p=0.9,
                                 pad_token_id=tokenizer.eos_token_id)
        print("[tuned   ]", decode(out, ids))
    except Exception as exc:
        print(f"[tuned   ] FAILED: {exc!r}")


## 6. Package the adapters for use

The `attackers/` directory now contains three LoRA adapter folders. Zip them so they can be downloaded and re-uploaded into the defender notebook (or included in the final submission zip).

## 7. Push adapters to Hugging Face Hub

So that the v2/v3 defender notebooks can fetch them automatically — no zip-upload dance, works on Colab and Kaggle alike.

This cell pushes each LoRA adapter folder as its own HF model repo.


In [ ]:
# Push each LoRA adapter to HF Hub so v2/v3 can pull them via snapshot_download.
from huggingface_hub import HfApi, login, notebook_login

# Pick up HF token from Kaggle Secrets if available, else interactive login.
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    notebook_login()

api = HfApi()
HF_USER = "NDGCodes"   # change if pushing to a different account

for persona in ("authority", "consensus", "gaslighter"):
    repo_id  = f"{HF_USER}/sia-{persona}-lora"
    local_dir = f"{ATTACKERS_DIR}/{persona}_lora"
    print(f"Pushing {local_dir} → {repo_id}")
    api.create_repo(repo_id=repo_id, repo_type="model",
                    exist_ok=True, private=False)
    api.upload_folder(
        folder_path=local_dir, repo_id=repo_id, repo_type="model",
        commit_message="LoRA adapter for Social Influence Arena attacker persona",
    )
print("\u2705 All three adapters pushed to HF Hub")


In [ ]:
import shutil, os
shutil.make_archive(f"{WORKDIR}/attacker_adapters", "zip", "/kaggle/working", "attackers")
print("Adapters zipped → /kaggle/working/attacker_adapters.zip")
print("Download the zip from the Kaggle output panel, then re-upload the whole thing into the main siarena2.ipynb input so its LLMAttackerPanel eval can find it.")